# Segmentación ciudadana de participación electoral

Perfiles nacionales de participación a partir de celdas demográficas
(sección x sexo x grupo de edad) usando clustering sobre indicadores
de comportamiento electoral 2009-2024.

Pipeline:
1. Carga de celdas con historial de participación y peso poblacional.
2. Clustering nacional (K-Means ponderado por lista nominal).
3. Caracterización de perfiles: trayectoria, demografia, geografia, PJ 2025.
4. Mapa nacional interactivo con 3 niveles de agregación (estatal, municipal, clustering local).

## 1. Configuración

In [ ]:
import warnings
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyogrio
from branca.element import Template, MacroElement
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

## 2. Parametros y rutas

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DIR_PREPROCESSED = ROOT / "DATA" / "PREPROCESSED"
DIR_GEO = ROOT / "DATA" / "RAW" / "marco_geoestadistico"
DIR_OUTPUT = DIR_PREPROCESSED

FEATURES = ["promedio", "brecha_pres", "tendencia"]
ELECCIONES = ["P2009", "P2012", "P2015", "P2018", "P2021", "P2024"]
CORTES_EDAD = [17, 24, 29, 39, 49, 59, 69, 79, 200]
ETIQUETAS_EDAD = ["18-24", "25-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
K_RANGE = range(3, 8)
K_OBLIGATORIO = range(4, 7)  # convocatoria pide 4-6 perfiles
MIN_CELDAS_MUNI = 30
RANDOM_STATE = 42
print(f"ROOT: {ROOT}")

## 3. Carga de datos

### 3.1 Celdas con historial de participación

657k celdas (sección x sexo x grupo de edad) con tasas de participación
en las 6 elecciones federales 2009-2024 y 3 indicadores derivados.

In [ ]:
celdas = pd.read_parquet(DIR_PREPROCESSED / "celdas_historial.parquet")
print(f"Celdas: {len(celdas):,}")
print(f"Columnas: {list(celdas.columns)}")

### 3.2 Peso poblacional por celda

Se usa la lista nominal 2024 (SV + NV) agregada por celda como peso.
Esto hace que los perfiles representen personas, no celdas estadísticas.

In [ ]:
fed24 = pd.read_parquet(
    DIR_PREPROCESSED / "conteos_federal_2009_2024.parquet",
    columns=["EDOCVE", "SECCION", "SEXO", "EDAD", "AELEC", "SV", "NV",
             "MPIOCVE", "MPIONOM"],
    filters=[("AELEC", "==", 2024)],
)

fed24["GRUPO_EDAD"] = pd.cut(
    fed24["EDAD"], bins=CORTES_EDAD, labels=ETIQUETAS_EDAD
)
fed24["VOTANTES"] = fed24["SV"] + fed24["NV"]

pesos = (
    fed24.groupby(["EDOCVE", "SECCION", "SEXO", "GRUPO_EDAD"], observed=True)["VOTANTES"]
    .sum()
    .reset_index()
    .rename(columns={"VOTANTES": "peso"})
)
pesos["GRUPO_EDAD"] = pesos["GRUPO_EDAD"].astype(str)

sec_mpio = (
    fed24.groupby(["EDOCVE", "SECCION"])
    .agg(MPIOCVE=("MPIOCVE", "first"), MPIONOM=("MPIONOM", "first"))
    .reset_index()
)
del fed24

celdas["GRUPO_EDAD"] = celdas["GRUPO_EDAD"].astype(str)
celdas = celdas.merge(pesos, on=["EDOCVE", "SECCION", "SEXO", "GRUPO_EDAD"], how="left")
sin_peso = celdas["peso"].isna().sum()
if sin_peso:
    print(f"Celdas sin peso: {sin_peso} ({sin_peso/len(celdas)*100:.2f}%) - se asigna peso 1")
    celdas["peso"] = celdas["peso"].fillna(1)
celdas["peso"] = celdas["peso"].astype(int)

print(f"Peso total (personas): {celdas['peso'].sum():,}")
print(f"Peso medio por celda: {celdas['peso'].mean():.1f}")
print(f"Peso mediano: {celdas['peso'].median():.0f}")

### 3.3 Participación PJ 2025

Se agrega la tasa de participación en la elección del Poder Judicial 2025 por sección.
Correlación baja (0.28) con el promedio federal: es una dimensión distinta
que enriquece la caracterización de los perfiles.

In [ ]:
secciones = pd.read_parquet(
    DIR_PREPROCESSED / "secciones_historial.parquet",
    columns=["EDOCVE", "SECCION", "P_PJ2025", "LN2024"],
)
celdas = celdas.merge(secciones[["EDOCVE", "SECCION", "P_PJ2025"]], on=["EDOCVE", "SECCION"], how="left")
print(f"Celdas con P_PJ2025: {celdas['P_PJ2025'].notna().sum():,} de {len(celdas):,}")
del secciones

## 4. Clustering nacional

### 4.1 Preparacion

Features de comportamiento electoral (no demográficas):
- **promedio**: nivel general de participación (media 2009-2024)
- **brecha_pres**: sensibilidad al tipo de elección (presidencial - intermedia)
- **tendencia**: trayectoria temporal (pendiente en puntos por elección)

Las variables demográficas (edad, sexo, región) se usan después
para *caracterizar* los perfiles, no para definirlos.

In [ ]:
X = celdas[FEATURES].values
W = celdas["peso"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Celdas a clusterizar: {X_scaled.shape[0]:,}")
print(f"Medias originales: {dict(zip(FEATURES, X.mean(axis=0).round(2)))}")

### 4.2 Evaluación de k

Se evalua k = 3 a 7. La convocatoria pide 4-6 perfiles,
así que se selecciona el mejor silhouette dentro de ese rango.

In [ ]:
resultados_k = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    km.fit(X_scaled, sample_weight=W)
    labels = km.labels_
    sample_idx = np.random.RandomState(RANDOM_STATE).choice(
        len(X_scaled), size=min(50_000, len(X_scaled)), replace=False
    )
    sil = silhouette_score(X_scaled[sample_idx], labels[sample_idx])
    resultados_k.append({"k": k, "silhouette": sil, "inertia": km.inertia_})
    marcador = " *" if k in K_OBLIGATORIO else ""
    print(f"k={k}: silhouette={sil:.4f}, inertia={km.inertia_:,.0f}{marcador}")

df_k = pd.DataFrame(resultados_k)
candidatos = df_k[df_k.k.isin(K_OBLIGATORIO)]
mejor_k = int(candidatos.loc[candidatos.silhouette.idxmax(), "k"])
print(f"\nMejor k en rango 4-6: {mejor_k}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(df_k.k, df_k.silhouette, "o-", color="#333")
ax1.axvspan(K_OBLIGATORIO.start, K_OBLIGATORIO.stop - 1, alpha=0.1, color="blue")
ax1.axvline(mejor_k, color="red", linestyle="--", alpha=0.5)
ax1.set_xlabel("k")
ax1.set_ylabel("Silhouette")
ax1.set_title("Silhouette vs k")

ax2.plot(df_k.k, df_k.inertia, "o-", color="#333")
ax2.axvspan(K_OBLIGATORIO.start, K_OBLIGATORIO.stop - 1, alpha=0.1, color="blue")
ax2.axvline(mejor_k, color="red", linestyle="--", alpha=0.5)
ax2.set_xlabel("k")
ax2.set_ylabel("Inercia")
ax2.set_title("Inercia vs k")

plt.suptitle("Zona azul: rango 4-6 requerido por la convocatoria", fontsize=10, color="#666")
plt.tight_layout()
plt.show()

### 4.3 Modelo final

In [ ]:
km_final = KMeans(n_clusters=mejor_k, n_init=20, random_state=RANDOM_STATE)
km_final.fit(X_scaled, sample_weight=W)
labels_raw = km_final.labels_

centros_orig = scaler.inverse_transform(km_final.cluster_centers_)
orden = np.argsort(centros_orig[:, 0])
mapping = {old: new for new, old in enumerate(orden)}
celdas["perfil"] = np.array([mapping[l] for l in labels_raw])

print(f"Perfiles nacionales (k={mejor_k}):")
for p in range(mejor_k):
    mask = celdas.perfil == p
    w_p = celdas.loc[mask, "peso"]
    n_celdas = mask.sum()
    n_personas = w_p.sum()
    vals = np.average(celdas.loc[mask, FEATURES].values, axis=0, weights=w_p)
    print(f"  Perfil {p}: {n_celdas:,} celdas, {n_personas:,} personas ({n_personas/W.sum()*100:.1f}%)")
    print(f"    promedio={vals[0]:.1f}  brecha={vals[1]:+.1f}  tendencia={vals[2]:+.2f}")

## 5. Caracterización de perfiles

### 5.1 Resumen ponderado

Cada estadística se pondera por lista nominal 2024 (peso),
de modo que los porcentajes representan personas, no celdas.

In [ ]:
def wmean(df, col, wcol="peso"):
    return np.average(df[col], weights=df[wcol])

resumen_rows = []
for p in range(mejor_k):
    dp = celdas[celdas.perfil == p]
    w = dp["peso"]
    n_personas = int(w.sum())
    row = {
        "perfil": p,
        "n_celdas": len(dp),
        "n_personas": n_personas,
        "pct_poblacion": n_personas / celdas["peso"].sum() * 100,
        "promedio": wmean(dp, "promedio"),
        "brecha_pres": wmean(dp, "brecha_pres"),
        "tendencia": wmean(dp, "tendencia"),
        "pct_mujer": np.average(dp["SEXO"] == 1, weights=w) * 100,
    }
    for e in ELECCIONES:
        row[e] = wmean(dp, e)
    if dp["P_PJ2025"].notna().any():
        dp_pj = dp[dp["P_PJ2025"].notna()]
        row["P_PJ2025"] = wmean(dp_pj, "P_PJ2025")
    resumen_rows.append(row)

resumen = pd.DataFrame(resumen_rows)
cols_show = ["perfil", "n_personas", "pct_poblacion", "promedio", "brecha_pres", "tendencia", "pct_mujer", "P_PJ2025"]
print(resumen[cols_show].to_string(index=False, float_format=lambda x: f"{x:.1f}"))

### 5.2 Trayectorias de participación

Participación ponderada por elección para cada perfil.
Las presidenciales (2012, 2018, 2024) tienen picos; las intermedias valles.
La forma de la curva diferencia a los perfiles.

In [ ]:
COLORES_PERFIL = ["#3288bd", "#66c2a5", "#fee08b", "#fdae61", "#f46d43", "#d53e4f"]
anios = [2009, 2012, 2015, 2018, 2021, 2024]

fig, ax = plt.subplots(figsize=(10, 5))
for p in range(mejor_k):
    tray = [resumen.loc[p, e] for e in ELECCIONES]
    ax.plot(anios, tray, "o-", color=COLORES_PERFIL[p], linewidth=2, markersize=6,
            label=f"Perfil {p} ({resumen.loc[p, 'pct_poblacion']:.0f}%)")

ax.set_xlabel("Elección")
ax.set_ylabel("% Participación (ponderada)")
ax.set_title("Trayectoria de participación por perfil")
ax.legend(loc="upper left")
ax.set_xticks(anios)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 Composición demográfica

Distribución de edad y sexo dentro de cada perfil, ponderada por peso.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Edad
ax = axes[0]
edad_data = []
for p in range(mejor_k):
    dp = celdas[celdas.perfil == p]
    for g in ETIQUETAS_EDAD:
        mask_g = dp["GRUPO_EDAD"] == g
        pct = dp.loc[mask_g, "peso"].sum() / dp["peso"].sum() * 100
        edad_data.append({"perfil": p, "grupo": g, "pct": pct})
df_edad = pd.DataFrame(edad_data)

x = np.arange(len(ETIQUETAS_EDAD))
width = 0.8 / mejor_k
for p in range(mejor_k):
    vals = df_edad[df_edad.perfil == p]["pct"].values
    ax.bar(x + p * width, vals, width, color=COLORES_PERFIL[p], label=f"Perfil {p}")
ax.set_xticks(x + width * (mejor_k - 1) / 2)
ax.set_xticklabels(ETIQUETAS_EDAD, rotation=45)
ax.set_ylabel("% del perfil")
ax.set_title("Distribución de edad por perfil")
ax.legend(fontsize=8)

# Sexo
ax = axes[1]
for p in range(mejor_k):
    dp = celdas[celdas.perfil == p]
    pct_m = np.average(dp["SEXO"] == 1, weights=dp["peso"]) * 100
    pct_h = 100 - pct_m
    ax.barh(p, pct_h, color="#4a90d9", height=0.6)
    ax.barh(p, -pct_m, color="#d94a6b", height=0.6)
    ax.text(pct_h / 2, p, f"H {pct_h:.0f}%", ha="center", va="center", fontsize=9, color="white")
    ax.text(-pct_m / 2, p, f"M {pct_m:.0f}%", ha="center", va="center", fontsize=9, color="white")
ax.set_yticks(range(mejor_k))
ax.set_yticklabels([f"Perfil {p}" for p in range(mejor_k)])
ax.set_title("Composición por sexo")
ax.set_xlim(-70, 70)
ax.axvline(0, color="#333", linewidth=0.5)
ax.set_xlabel("% del perfil")

plt.tight_layout()
plt.show()

### 5.4 Distribución geográfica

Región y tipo de sección (urbana/mixta/rural) por perfil.

In [ ]:
REGIONES_ORDEN = ["Noroeste", "Noreste", "Occidente", "Centro", "Sur-Sureste"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Región
ax = axes[0]
reg_data = []
for p in range(mejor_k):
    dp = celdas[celdas.perfil == p]
    for r in REGIONES_ORDEN:
        mask_r = dp["REGION"] == r
        pct = dp.loc[mask_r, "peso"].sum() / dp["peso"].sum() * 100
        reg_data.append({"perfil": p, "región": r, "pct": pct})
df_reg = pd.DataFrame(reg_data)

x = np.arange(len(REGIONES_ORDEN))
width = 0.8 / mejor_k
for p in range(mejor_k):
    vals = df_reg[df_reg.perfil == p]["pct"].values
    ax.bar(x + p * width, vals, width, color=COLORES_PERFIL[p], label=f"Perfil {p}")
ax.set_xticks(x + width * (mejor_k - 1) / 2)
ax.set_xticklabels(REGIONES_ORDEN, rotation=30)
ax.set_ylabel("% del perfil")
ax.set_title("Distribución regional por perfil")
ax.legend(fontsize=8)

# Tipo de sección
ax = axes[1]
tipos = ["U", "M", "R"]
tipo_labels = ["Urbana", "Mixta", "Rural"]
tipo_data = []
for p in range(mejor_k):
    dp = celdas[celdas.perfil == p]
    for t in tipos:
        mask_t = dp["TIPOSEC"] == t
        pct = dp.loc[mask_t, "peso"].sum() / dp["peso"].sum() * 100
        tipo_data.append({"perfil": p, "tipo": t, "pct": pct})
df_tipo = pd.DataFrame(tipo_data)

x = np.arange(len(tipos))
width = 0.8 / mejor_k
for p in range(mejor_k):
    vals = df_tipo[df_tipo.perfil == p]["pct"].values
    ax.bar(x + p * width, vals, width, color=COLORES_PERFIL[p], label=f"Perfil {p}")
ax.set_xticks(x + width * (mejor_k - 1) / 2)
ax.set_xticklabels(tipo_labels)
ax.set_ylabel("% del perfil")
ax.set_title("Tipo de sección por perfil")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### 5.5 Participación en elección del Poder Judicial 2025

La elección judicial tuvo 13% de participación nacional (vs 60% en federales).
La correlación con el promedio federal es solo 0.28: captura una dimensión distinta.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for p in range(mejor_k):
    dp = celdas[(celdas.perfil == p) & celdas.P_PJ2025.notna()]
    if len(dp) == 0:
        continue
    pj = wmean(dp, "P_PJ2025")
    ax.barh(p, pj, color=COLORES_PERFIL[p], height=0.6)
    ax.text(pj + 0.3, p, f"{pj:.1f}%", va="center", fontsize=10)

ax.set_yticks(range(mejor_k))
ax.set_yticklabels([f"Perfil {p}" for p in range(mejor_k)])
ax.set_xlabel("% Participación PJ 2025 (ponderada)")
ax.set_title("Participación en elección del Poder Judicial 2025 por perfil")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

### 5.6 Nombres de perfil

Se asigna un nombre descriptivo a cada perfil basado en sus caracteristicas
dominantes. Los nombres deben ser accionables para el INE.

In [ ]:
nombres_perfil = {}
for p in range(mejor_k):
    dp = celdas[celdas.perfil == p]
    w = dp["peso"]
    prom = wmean(dp, "promedio")
    brecha = wmean(dp, "brecha_pres")
    tend = wmean(dp, "tendencia")
    pct_mujer = np.average(dp["SEXO"] == 1, weights=w) * 100
    edad_pesos = {g: dp.loc[dp["GRUPO_EDAD"] == g, "peso"].sum() for g in ETIQUETAS_EDAD}
    edad_dom = max(edad_pesos, key=edad_pesos.get)
    pct_rural = np.average(dp["TIPOSEC"].isin(["R", "M"]), weights=w) * 100

    parts = []

    if prom < 45:
        parts.append("Baja participación")
    elif prom < 55:
        parts.append("Participación moderada")
    elif prom < 65:
        parts.append("Participación media-alta")
    else:
        parts.append("Alta participación")

    if abs(brecha) > 15:
        parts.append("muy sensible a presidenciales")
    elif abs(brecha) < 8:
        parts.append("estable entre elecciones")

    if tend < -1:
        parts.append("en descenso")
    elif tend > 3:
        parts.append("en ascenso")

    nombre = ", ".join(parts)
    nombres_perfil[p] = nombre
    print(f"Perfil {p}: {nombre}")
    print(f"  prom={prom:.1f}% brecha={brecha:+.1f} tend={tend:+.2f}")
    print(f"  {pct_mujer:.0f}% mujeres, edad dominante {edad_dom}, {pct_rural:.0f}% rural+mixta")
    print()

### 5.7 Fichas de perfil

Resumen visual consolidado por perfil. Cada ficha reune comportamiento electoral,
composición demográfica y distribución geográfica en una sola página. Las barras
grises muestran el promedio nacional como referencia; los números entre parentesis
indican la diferencia respecto al nacional.

**Nota:** Los datos del CCPC no incluyen escolaridad, ingreso ni autoadscripcion
indígena/afromexicana (esta última solo aparece como filtro en los tableros
interactivos, no en los CSVs descargados). Las dimensiones presentadas son todas
las disponibles en los microdatos.

In [ ]:
EDONOM = {
    1: "Aguascalientes", 2: "Baja California", 3: "Baja California Sur",
    4: "Campeche", 5: "Coahuila", 6: "Colima", 7: "Chiapas", 8: "Chihuahua",
    9: "Ciudad de México", 10: "Durango", 11: "Guanajuato", 12: "Guerrero",
    13: "Hidalgo", 14: "Jalisco", 15: "México", 16: "Michoacán", 17: "Morelos",
    18: "Nayarit", 19: "Nuevo Leon", 20: "Oaxaca", 21: "Puebla", 22: "Queretaro",
    23: "Quintana Roo", 24: "San Luis Potosi", 25: "Sinaloa", 26: "Sonora",
    27: "Tabasco", 28: "Tamaulipas", 29: "Tlaxcala", 30: "Veracruz",
    31: "Yucatan", 32: "Zacatecas",
}

w_all = celdas["peso"]
nac_tray = [float(np.average(celdas[e], weights=w_all)) for e in ELECCIONES]
nac_edad = [float(celdas.loc[celdas.GRUPO_EDAD == g, "peso"].sum() / w_all.sum() * 100)
            for g in ETIQUETAS_EDAD]
nac_reg = [float(celdas.loc[celdas.REGION == r, "peso"].sum() / w_all.sum() * 100)
           for r in REGIONES_ORDEN]
print("Referencias nacionales calculadas.")

In [ ]:
for p in range(mejor_k):
    dp = celdas[celdas.perfil == p]
    w = dp["peso"]
    n_pers = int(w.sum())
    pct_pob = n_pers / w_all.sum() * 100
    color = COLORES_PERFIL[p]

    fig = plt.figure(figsize=(16, 10), facecolor="white")
    gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.4,
                          top=0.87, bottom=0.07, left=0.06, right=0.97)
    fig.suptitle(f"Perfil {p}: {nombres_perfil[p]}",
                 fontsize=15, fontweight="bold", color=color, y=0.96)
    fig.text(0.5, 0.915, f"{n_pers:,} personas  ({pct_pob:.1f}%)",
             ha="center", fontsize=12, color="#555")

    # --- Trayectoria (2 columnas) ---
    ax = fig.add_subplot(gs[0, :2])
    tray = [resumen.loc[p, e] for e in ELECCIONES]
    ax.plot(anios, nac_tray, "s--", color="#bbb", markersize=4, linewidth=1.5,
            label="Nacional", zorder=2)
    ax.plot(anios, tray, "o-", color=color, linewidth=2.5, markersize=8,
            label=f"Perfil {p}", zorder=3)
    for a, v in zip(anios, tray):
        ax.annotate(f"{v:.1f}", (a, v), textcoords="offset points",
                    xytext=(0, 10), ha="center", fontsize=9, fontweight="bold")
    ax.fill_between(anios, tray, alpha=0.08, color=color)
    ax.set_ylabel("% Participación")
    ax.set_title("Trayectoria 2009-2024  (gris = promedio nacional)", fontsize=11)
    ax.set_xticks(anios)
    ax.grid(axis="y", alpha=0.3)
    lo = min(min(tray), min(nac_tray)) - 6
    hi = max(max(tray), max(nac_tray)) + 6
    ax.set_ylim(max(0, lo), min(100, hi))
    ax.legend(fontsize=9, loc="lower right")

    # --- Panel de métricas ---
    ax = fig.add_subplot(gs[0, 2])
    ax.axis("off")
    prom = wmean(dp, "promedio")
    brecha = wmean(dp, "brecha_pres")
    tend = wmean(dp, "tendencia")
    pct_mujer = np.average(dp.SEXO == 1, weights=w) * 100
    pct_hombre = 100 - pct_mujer
    pct_u = np.average(dp.TIPOSEC == "U", weights=w) * 100
    pct_mx = np.average(dp.TIPOSEC == "M", weights=w) * 100
    pct_r = np.average(dp.TIPOSEC == "R", weights=w) * 100
    dp_pj = dp[dp.P_PJ2025.notna()]
    pj = wmean(dp_pj, "P_PJ2025") if len(dp_pj) else float("nan")

    items = [
        ("PARTICIPACIÓN", None, True),
        ("Promedio", f"{prom:.1f}%", False),
        ("Brecha pres.", f"{brecha:+.1f} pts", False),
        ("Tendencia", f"{tend:+.2f} pts/elec", False),
        ("GÉNERO", None, True),
        ("Hombres", f"{pct_hombre:.0f}%", False),
        ("Mujeres", f"{pct_mujer:.0f}%", False),
        ("TIPO DE SECCION", None, True),
        ("Urbana", f"{pct_u:.0f}%", False),
        ("Mixta", f"{pct_mx:.0f}%", False),
        ("Rural", f"{pct_r:.0f}%", False),
        ("JUDICIAL 2025", None, True),
        ("Participación", f"{pj:.1f}%", False),
    ]
    y = 0.98
    for label, value, is_header in items:
        if is_header:
            y -= 0.005
            ax.text(0.05, y, label, transform=ax.transAxes,
                    fontsize=8, fontweight="bold", color="#aaa",
                    fontstyle="italic")
            y -= 0.055
        else:
            ax.text(0.12, y, label, transform=ax.transAxes,
                    fontsize=10, color="#555")
            ax.text(0.95, y, value, transform=ax.transAxes,
                    fontsize=11, fontweight="bold", ha="right", color="#333")
            y -= 0.065

    # --- Edad ---
    ax = fig.add_subplot(gs[1, 0])
    edad_pcts = [dp.loc[dp.GRUPO_EDAD == g, "peso"].sum() / w.sum() * 100
                 for g in ETIQUETAS_EDAD]
    yp = np.arange(len(ETIQUETAS_EDAD))
    ax.barh(yp, nac_edad, color="#e0e0e0", height=0.7)
    ax.barh(yp, edad_pcts, color=color, alpha=0.8, height=0.7)
    for i, (v, n) in enumerate(zip(edad_pcts, nac_edad)):
        diff = v - n
        sign = "+" if diff >= 0 else ""
        ax.text(max(v, n) + 0.3, i, f"{v:.1f} ({sign}{diff:.1f})",
                va="center", fontsize=8)
    ax.set_yticks(yp)
    ax.set_yticklabels(ETIQUETAS_EDAD)
    ax.set_title("Edad  (gris = nacional)", fontsize=11)
    ax.set_xlabel("% del perfil")
    ax.invert_yaxis()

    # --- Región ---
    ax = fig.add_subplot(gs[1, 1])
    reg_pcts = [dp.loc[dp.REGION == r, "peso"].sum() / w.sum() * 100
                for r in REGIONES_ORDEN]
    yp = np.arange(len(REGIONES_ORDEN))
    ax.barh(yp, nac_reg, color="#e0e0e0", height=0.6)
    ax.barh(yp, reg_pcts, color=color, alpha=0.8, height=0.6)
    for i, (v, n) in enumerate(zip(reg_pcts, nac_reg)):
        diff = v - n
        sign = "+" if diff >= 0 else ""
        ax.text(max(v, n) + 0.3, i, f"{v:.1f} ({sign}{diff:.1f})",
                va="center", fontsize=8)
    ax.set_yticks(yp)
    ax.set_yticklabels(REGIONES_ORDEN)
    ax.set_title("Región  (gris = nacional)", fontsize=11)
    ax.set_xlabel("% del perfil")
    ax.invert_yaxis()

    # --- Top 5 entidades ---
    ax = fig.add_subplot(gs[1, 2])
    edo_w = dp.groupby("EDOCVE")["peso"].sum()
    edo_pct = (edo_w / w.sum() * 100).sort_values(ascending=False).head(5)
    labels = [EDONOM.get(int(e), str(e))[:20] for e in edo_pct.index]
    yp = np.arange(len(labels))
    ax.barh(yp, edo_pct.values, color=color, alpha=0.8, height=0.6)
    for i, v in enumerate(edo_pct.values):
        ax.text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=9)
    ax.set_yticks(yp)
    ax.set_yticklabels(labels)
    ax.set_title("Top 5 entidades", fontsize=11)
    ax.set_xlabel("% del perfil")
    ax.invert_yaxis()

    plt.show()
    print()

## 6. Mapa nacional interactivo

Tres niveles de agregación:
1. **Estatal:** perfil dominante y distribución por entidad.
2. **Municipal:** perfil dominante por municipio, coloreado sobre el mapa.
3. **Clustering municipal:** segmentación local con detalle demográfico en popup.

### 6.1 Geometrias

In [ ]:
gdf_mun = pyogrio.read_dataframe(DIR_GEO / "mun22gw.shp", encoding="latin-1")
gdf_mun["EDOCVE"] = gdf_mun["CVE_ENT"].astype(int)
gdf_mun["MPIOCVE"] = gdf_mun["CVE_MUN"].astype(int)
gdf_mun["geometry"] = gdf_mun.geometry.simplify(tolerance=0.008, preserve_topology=True)
print(f"Municipios nacional: {len(gdf_mun)}")

gdf_edo = gdf_mun.dissolve(by="CVE_ENT").reset_index()
gdf_edo["EDOCVE"] = gdf_edo["CVE_ENT"].astype(int)
gdf_edo["geometry"] = gdf_edo.geometry.simplify(tolerance=0.01, preserve_topology=True)
print(f"Entidades: {len(gdf_edo)}")

### 6.2 Asignación de municipio a todas las celdas

In [ ]:
celdas_geo = celdas.merge(sec_mpio[["EDOCVE", "SECCION", "MPIOCVE", "MPIONOM"]], on=["EDOCVE", "SECCION"], how="left")
sin_mpio = celdas_geo.MPIOCVE.isna().sum()
if sin_mpio:
    print(f"Celdas sin municipio: {sin_mpio} ({sin_mpio/len(celdas_geo)*100:.1f}%) - se descartan para el mapa")
    celdas_geo = celdas_geo.dropna(subset=["MPIOCVE"])
celdas_geo["MPIOCVE"] = celdas_geo["MPIOCVE"].astype(int)
print(f"Celdas geolocalizadas: {len(celdas_geo):,}")
print(f"Municipios: {celdas_geo.MPIOCVE.nunique()}, Entidades: {celdas_geo.EDOCVE.nunique()}")

### 6.3 Agregación estatal

In [ ]:
EDONOM_MAP = dict(zip(
    gdf_mun["EDOCVE"], gdf_mun["NOM_ENT"]
))

stats_edo = {}
for edocve, df_e in celdas_geo.groupby("EDOCVE"):
    w = df_e["peso"]
    perfil_dist = {}
    for p in range(mejor_k):
        mask_p = df_e["perfil"] == p
        perfil_dist[p] = float(df_e.loc[mask_p, "peso"].sum() / w.sum())
    prom = float(np.average(df_e["promedio"], weights=w))
    stats_edo[edocve] = {
        "nombre": EDONOM_MAP.get(edocve, str(edocve)),
        "n_celdas": len(df_e),
        "n_personas": int(w.sum()),
        "perfil_dist": perfil_dist,
        "promedio": prom,
    }
print(f"Entidades con datos: {len(stats_edo)}")

### 6.4 Conteo y clustering municipal nacional

In [ ]:
conteo_mpio = (
    celdas_geo.groupby(["EDOCVE", "MPIOCVE", "MPIONOM"])
    .agg(n_celdas=("perfil", "size"), peso_total=("peso", "sum"))
    .reset_index()
)
conteo_mpio["suficiente"] = conteo_mpio.n_celdas >= MIN_CELDAS_MUNI

n_suf = conteo_mpio.suficiente.sum()
n_bloq = (~conteo_mpio.suficiente).sum()
print(f"Municipios suficientes: {n_suf} (>= {MIN_CELDAS_MUNI} celdas)")
print(f"Municipios bloqueados:  {n_bloq}")

In [ ]:
GRUPOS_EDAD_ORDEN = ETIQUETAS_EDAD

def elegir_k_municipal(n):
    if n < 100:
        return 2
    if n < 300:
        return 3
    return None

resultados_mpio = {}
mpios_suf = conteo_mpio[conteo_mpio.suficiente]

for _, row_m in mpios_suf.iterrows():
    edocve = int(row_m.EDOCVE)
    mpiocve = int(row_m.MPIOCVE)
    nombre = row_m.MPIONOM

    mask = (celdas_geo.EDOCVE == edocve) & (celdas_geo.MPIOCVE == mpiocve)
    df_m = celdas_geo.loc[mask]
    X_m = df_m[FEATURES].values
    W_m = df_m["peso"].values
    n_m = len(X_m)

    scaler_m = StandardScaler()
    X_ms = scaler_m.fit_transform(X_m)

    k_fijo = elegir_k_municipal(n_m)
    if k_fijo is not None:
        k_m = k_fijo
    else:
        mejor_sil = -1
        k_m = 3
        for k_cand in range(3, 6):
            km_cand = KMeans(n_clusters=k_cand, n_init=10, random_state=RANDOM_STATE)
            km_cand.fit(X_ms, sample_weight=W_m)
            sil = silhouette_score(X_ms, km_cand.labels_)
            if sil > mejor_sil:
                mejor_sil = sil
                k_m = k_cand

    km_m = KMeans(n_clusters=k_m, n_init=10, random_state=RANDOM_STATE)
    km_m.fit(X_ms, sample_weight=W_m)
    centros_m = scaler_m.inverse_transform(km_m.cluster_centers_)
    orden_m = np.argsort(centros_m[:, 0])
    mapping_m = {old: new for new, old in enumerate(orden_m)}
    labels_m_ord = np.array([mapping_m[l] for l in km_m.labels_])

    clusters_info = []
    for c in range(k_m):
        mask_c = labels_m_ord == c
        df_c = df_m.iloc[mask_c.nonzero()[0]]
        w_c = df_c["peso"]
        pct_mujer = float(np.average(df_c.SEXO == 1, weights=w_c) * 100)
        edad_dist = {g: float(df_c.loc[df_c.GRUPO_EDAD == g, "peso"].sum() / w_c.sum() * 100) for g in GRUPOS_EDAD_ORDEN}
        centro_c = centros_m[orden_m[c]]
        clusters_info.append({
            "cluster": c, "n": int(mask_c.sum()), "pct": float(mask_c.sum() / n_m * 100),
            "promedio": float(centro_c[0]), "brecha_pres": float(centro_c[1]),
            "tendencia": float(centro_c[2]),
            "pct_hombre": 100 - pct_mujer, "pct_mujer": pct_mujer,
            "edad_dist": edad_dist,
        })

    perfil_dist = {}
    for p in range(mejor_k):
        mask_p = df_m["perfil"] == p
        perfil_dist[p] = float(df_m.loc[mask_p, "peso"].sum() / df_m["peso"].sum())

    resultados_mpio[(edocve, mpiocve)] = {
        "nombre": nombre, "n_celdas": n_m, "k": k_m,
        "clusters": clusters_info, "perfil_dist": perfil_dist,
    }

print(f"Municipios con clustering: {len(resultados_mpio)}")

### 6.5 Construir mapa

El mapa no se despliega en el notebook (pesaria ~54 MB embebido). Se abre desde `DATA/PREPROCESSED/mapa_nacional_segmentacion.html`, que se guarda en la siguiente celda.

In [ ]:
COLORES_CLUSTER_MUNI = ["#1b7837", "#5aae61", "#d9a441", "#c2524a", "#8c510a"]
COLOR_BLOQUEADO = "#cccccc"

perfil_info = {}
for p in range(mejor_k):
    dp = celdas[celdas.perfil == p]
    w = dp["peso"]
    pct_mujer = float(np.average(dp.SEXO == 1, weights=w) * 100)
    edad_dist = {}
    for g in ETIQUETAS_EDAD:
        edad_dist[g] = float(dp.loc[dp.GRUPO_EDAD == g, "peso"].sum() / w.sum() * 100)
    pct_u = float(np.average(dp.TIPOSEC == "U", weights=w) * 100)
    pct_mx = float(np.average(dp.TIPOSEC == "M", weights=w) * 100)
    pct_r = float(np.average(dp.TIPOSEC == "R", weights=w) * 100)
    perfil_info[p] = {
        "nombre": nombres_perfil[p],
        "n_personas": int(w.sum()),
        "pct_pob": float(w.sum() / celdas["peso"].sum() * 100),
        "promedio": float(resumen.loc[p, "promedio"]),
        "brecha_pres": float(resumen.loc[p, "brecha_pres"]),
        "tendencia": float(resumen.loc[p, "tendencia"]),
        "pct_hombre": 100 - pct_mujer,
        "pct_mujer": pct_mujer,
        "edad_dist": edad_dist,
        "pct_urbana": pct_u,
        "pct_mixta": pct_mx,
        "pct_rural": pct_r,
    }
print(f"Info de {len(perfil_info)} perfiles calculada para popups.")

In [ ]:
def barra_sexo_html(pct_h, pct_m):
    return (
        f'<div style="display:flex;height:14px;border-radius:3px;overflow:hidden;'
        f'margin:2px 0 4px 0;font-size:10px;line-height:14px;color:#fff;">'
        f'<div style="width:{pct_h:.0f}%;background:#4a90d9;text-align:center;">H {pct_h:.0f}%</div>'
        f'<div style="width:{pct_m:.0f}%;background:#d94a6b;text-align:center;">M {pct_m:.0f}%</div></div>'
    )


def barras_edad_html(edad_dist, color_bar="#7b9fc2"):
    max_pct = max(edad_dist.values()) if edad_dist else 1
    html = '<table style="width:100%;border-collapse:collapse;margin:2px 0;">'
    for grupo, pct in edad_dist.items():
        w = max(1, int(pct / max_pct * 100))
        html += (
            f'<tr><td style="width:40px;font-size:10px;padding:1px 4px 1px 0;text-align:right;">{grupo}</td>'
            f'<td><div style="background:{color_bar};height:10px;width:{w}px;border-radius:2px;"></div></td>'
            f'<td style="width:30px;font-size:10px;padding-left:4px;">{pct:.0f}%</td></tr>'
        )
    html += '</table>'
    return html


def detalle_perfil_html(p_idx, pct_local):
    pi = perfil_info[p_idx]
    c = COLORES_PERFIL[p_idx]
    html = (
        f'<div style="border-left:4px solid {c};padding-left:6px;margin:4px 0 8px 0;">'
        f'<b style="color:{c};">Perfil {p_idx}</b> \u2014 '
        f'<b>{pct_local:.1f}%</b> local<br>'
        f'<span style="font-size:10px;color:#666;">{pi["nombre"]}</span><br>'
        f'<span style="font-size:10px;">Prom: {pi["promedio"]:.1f}% '
        f'Brecha: {pi["brecha_pres"]:+.1f} '
        f'Tend: {pi["tendencia"]:+.2f}</span><br>'
        f'<span style="font-size:10px;font-weight:bold;">Sexo:</span><br>'
    )
    html += barra_sexo_html(pi["pct_hombre"], pi["pct_mujer"])
    html += '<span style="font-size:10px;font-weight:bold;">Edad:</span><br>'
    html += barras_edad_html(pi["edad_dist"], color_bar=c)
    html += (
        f'<span style="font-size:10px;color:#888;">'
        f'Urb {pi["pct_urbana"]:.0f}% | Mix {pi["pct_mixta"]:.0f}% | Rur {pi["pct_rural"]:.0f}%'
        f'</span></div>'
    )
    return html


def popup_estado(edocve, s):
    html = (
        f'<div style="font-family:Arial;font-size:12px;width:420px;'
        f'max-height:550px;overflow-y:auto;">'
        f'<b style="font-size:14px;">{s["nombre"]}</b><br>'
        f'{s["n_celdas"]:,} celdas, {s["n_personas"]:,} personas<br>'
        f'Participación promedio: {s["promedio"]:.1f}%<br>'
        f'<hr style="border:none;border-top:1px solid #ccc;margin:6px 0;">'
        f'<b>Composición por perfil nacional</b><br>'
    )
    for p in range(mejor_k):
        pct = s["perfil_dist"].get(p, 0) * 100
        if pct > 0:
            html += detalle_perfil_html(p, pct)
    html += '</div>'
    return html


def popup_muni_suf(nombre, r):
    html = (
        f'<div style="font-family:Arial;font-size:12px;width:440px;'
        f'max-height:600px;overflow-y:auto;">'
        f'<b style="font-size:13px;">{nombre}</b><br>'
        f'{r["n_celdas"]:,} celdas<br>'
        f'<hr style="border:none;border-top:1px solid #ccc;margin:6px 0;">'
        f'<b>Perfiles nacionales en este municipio</b><br>'
    )
    for p in range(mejor_k):
        pct = r["perfil_dist"].get(p, 0) * 100
        if pct > 0:
            html += detalle_perfil_html(p, pct)
    html += '<hr style="border:none;border-top:1px solid #ccc;margin:6px 0;">'
    html += f'<b>Clustering local (k={r["k"]})</b><br>'
    for cl in r["clusters"]:
        ci = cl["cluster"]
        cc = COLORES_CLUSTER_MUNI[ci % len(COLORES_CLUSTER_MUNI)]
        html += (
            f'<div style="border-left:4px solid {cc};padding-left:6px;margin:6px 0;">'
            f'<b>C{ci}</b> {cl["n"]} ({cl["pct"]:.0f}%) '
            f'prom={cl["promedio"]:.1f} brecha={cl["brecha_pres"]:+.1f} tend={cl["tendencia"]:+.2f}<br>'
        )
        html += barra_sexo_html(cl["pct_hombre"], cl["pct_mujer"])
        html += barras_edad_html(cl["edad_dist"])
        html += '</div>'
    html += '</div>'
    return html


def popup_muni_bloq(nombre, n_celdas):
    return (
        f'<div style="font-family:Arial;font-size:12px;width:200px;">'
        f'<b>{nombre}</b><br>Celdas: {n_celdas}<br>'
        f'<span style="color:#999;">Muestra insuficiente</span></div>'
    )


print("Funciones de popup definidas.")

In [ ]:
m = folium.Map(location=[23.6, -102.5], zoom_start=5, tiles="CartoDB positron")

# --- Capa estatal ---
fg_edo = folium.FeatureGroup(name="Entidades", show=True)

for _, row in gdf_edo.iterrows():
    edocve = row.EDOCVE
    if edocve not in stats_edo:
        continue
    s = stats_edo[edocve]
    dom = max(s["perfil_dist"], key=s["perfil_dist"].get)
    color = COLORES_PERFIL[dom % len(COLORES_PERFIL)]

    geojson = folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda x, c=color: {
            "fillColor": c, "color": "#333", "weight": 2,
            "fillOpacity": 0.35,
        },
    )
    geojson.add_child(folium.Popup(popup_estado(edocve, s), max_width=450))
    geojson.add_to(fg_edo)

fg_edo.add_to(m)

# --- Capa municipal ---
fg_mun = folium.FeatureGroup(name="Municipios", show=False)

conteo_idx = conteo_mpio.set_index(["EDOCVE", "MPIOCVE"])

for _, row in gdf_mun.iterrows():
    edocve = row.EDOCVE
    mpiocve = row.MPIOCVE
    nombre = row.NOMGEO
    key = (edocve, mpiocve)

    if key in conteo_idx.index:
        n_c = int(conteo_idx.loc[key, "n_celdas"])
        suf = bool(conteo_idx.loc[key, "suficiente"])
    else:
        n_c = 0
        suf = False

    if suf and key in resultados_mpio:
        r = resultados_mpio[key]
        dom = max(r["perfil_dist"], key=r["perfil_dist"].get)
        color = COLORES_PERFIL[dom % len(COLORES_PERFIL)]
        popup_h = popup_muni_suf(nombre, r)
    else:
        color = COLOR_BLOQUEADO
        popup_h = popup_muni_bloq(nombre, n_c)

    geojson = folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda x, c=color, s=suf: {
            "fillColor": c, "color": "#666" if s else "#aaa",
            "weight": 0.5, "fillOpacity": 0.6 if s else 0.15,
        },
    )
    geojson.add_child(folium.Popup(popup_h, max_width=460))
    geojson.add_to(fg_mun)

fg_mun.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

# Panel de referencia de perfiles
panel_html = (
    '<div style="position:fixed;top:10px;right:10px;z-index:1000;'
    'background:white;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.3);'
    'font-family:Arial;font-size:11px;max-height:90vh;width:300px;overflow-y:auto;">'
    '<details open>'
    '<summary style="cursor:pointer;padding:10px 14px;font-weight:bold;font-size:13px;'
    'list-style:none;">Perfiles nacionales &#9660;</summary>'
    '<div style="padding:0 14px 10px 14px;">'
)
for p in range(mejor_k):
    pi = perfil_info[p]
    c = COLORES_PERFIL[p]
    panel_html += (
        f'<div style="border-left:4px solid {c};padding-left:8px;margin-bottom:14px;">'
        f'<b style="color:{c};font-size:12px;">Perfil {p}</b><br>'
        f'<span style="font-size:11px;font-weight:bold;">{pi["nombre"]}</span><br>'
        f'<span style="color:#666;">{pi["n_personas"]:,} personas ({pi["pct_pob"]:.1f}%)</span><br><br>'
        f'<span style="font-size:10px;">'
        f'Prom: <b>{pi["promedio"]:.1f}%</b> &nbsp; '
        f'Brecha: <b>{pi["brecha_pres"]:+.1f}</b> &nbsp; '
        f'Tend: <b>{pi["tendencia"]:+.2f}</b></span><br>'
    )
    panel_html += (
        f'<div style="display:flex;height:12px;border-radius:3px;overflow:hidden;'
        f'margin:4px 0;font-size:9px;line-height:12px;color:#fff;">'
        f'<div style="width:{pi["pct_hombre"]:.0f}%;background:#4a90d9;text-align:center;">'
        f'H {pi["pct_hombre"]:.0f}%</div>'
        f'<div style="width:{pi["pct_mujer"]:.0f}%;background:#d94a6b;text-align:center;">'
        f'M {pi["pct_mujer"]:.0f}%</div></div>'
    )
    ed = pi["edad_dist"]
    max_ed = max(ed.values()) if ed else 1
    panel_html += '<table style="width:100%;border-collapse:collapse;margin:3px 0;">'
    for g, pct in ed.items():
        bw = max(1, int(pct / max_ed * 90))
        panel_html += (
            f'<tr><td style="width:35px;font-size:9px;text-align:right;padding-right:3px;">{g}</td>'
            f'<td><div style="background:{c};opacity:0.7;height:8px;width:{bw}px;'
            f'border-radius:2px;"></div></td>'
            f'<td style="width:28px;font-size:9px;padding-left:3px;">{pct:.0f}%</td></tr>'
        )
    panel_html += '</table>'
    panel_html += (
        f'<span style="font-size:9px;color:#888;">'
        f'Urb {pi["pct_urbana"]:.0f}% | Mix {pi["pct_mixta"]:.0f}% | Rur {pi["pct_rural"]:.0f}%'
        f'</span></div>'
    )
panel_html += (
    f'<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eee;">'
    f'<span style="display:inline-block;width:10px;height:10px;'
    f'background:{COLOR_BLOQUEADO};margin-right:6px;border-radius:2px;"></span>'
    f'<span style="color:#999;">Sin clustering local</span></div>'
    f'</div></details></div>'
)
m.get_root().html.add_child(folium.Element(panel_html))


In [ ]:
ruta_html = DIR_OUTPUT / "mapa_nacional_segmentacion.html"
m.save(str(ruta_html))
print(f"Mapa guardado: {ruta_html}")
print(f"Tamaño: {ruta_html.stat().st_size / 1024:.0f} KB")

## 7. Exportacion

In [ ]:
celdas_out = celdas[
    ["EDOCVE", "SECCION", "SEXO", "GRUPO_EDAD", "peso"]
    + ELECCIONES + FEATURES + ["P_PJ2025", "perfil", "TIPOSEC", "REGION"]
].copy()
ruta = DIR_OUTPUT / "celdas_perfiladas.parquet"
celdas_out.to_parquet(ruta, index=False)
print(f"Celdas perfiladas: {ruta.name} ({len(celdas_out):,} filas)")

resumen_out = resumen.copy()
resumen_out["nombre"] = resumen_out["perfil"].map(nombres_perfil)
resumen_out.to_csv(DIR_OUTPUT / "resumen_perfiles.csv", index=False)
print(f"Resumen perfiles: resumen_perfiles.csv ({len(resumen_out)} filas)")

filas_res = []
for (edocve, mpiocve), r in resultados_mpio.items():
    dom = max(r["perfil_dist"], key=r["perfil_dist"].get)
    filas_res.append({
        "EDOCVE": edocve, "MPIOCVE": mpiocve, "municipio": r["nombre"],
        "n_celdas": r["n_celdas"], "k_local": r["k"],
        "perfil_dominante": dom,
        "pct_dominante": r["perfil_dist"][dom] * 100,
    })
resumen_mpio = pd.DataFrame(filas_res).sort_values("n_celdas", ascending=False)
resumen_mpio.to_csv(DIR_OUTPUT / "resumen_municipios.csv", index=False)
print(f"Resumen municipios: resumen_municipios.csv ({len(resumen_mpio)} filas)")

print("\nArchivos en DATA/PREPROCESSED/:")
for f in sorted(DIR_OUTPUT.iterdir()):
    if f.suffix in (".parquet", ".csv", ".html"):
        print(f"  {f.name}: {f.stat().st_size / 1024:.0f} KB")